## Simulate Ion Channel

This notebook loads an ion channel model from the Open Brain Platform and simulates it.

In [ ]:
import numpy
import pandas
import json

from ipywidgets import widgets, interact
from matplotlib import pyplot as plt

from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import SingleNeuronSimulation
from obi_notebook import get_projects
from obi_notebook import get_entities
from obi_notebook.get_environment import get_environment

## Reading in data

You will first have to authenticate to the platform. To do that, just click on the link after running the python cell below and follow instructions. Then, you will have to select in the drop-down menu the project that was used to produce the single cell simulations you want to analyse.

In [ ]:
token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects.get_projects(token)

In [ ]:
# get the suffix from the nmodl_suffix metadata.json file
import json
from neuron import h
import matplotlib.pyplot as plt

# file_path = "../ion-channel-model/metadata.json"
file_path="/Users/mandge/Desktop/ion-channel-model/metadata.json"
with open(file_path, "r") as f:
    metadata = json.load(f)
    suffix = metadata[0]["nmodl_suffix"]
    id = metadata[0]["id"]

In [ ]:
# channel_id="38a07b9f-e56d-4219-849c-d34ee4848f49"

In [ ]:
# download the ion channel model using entitysdk
from entitysdk.models import IonChannel
from entitysdk.client import Client

from obi_auth import get_token
from obi_notebook import get_projects
from obi_notebook import get_entities
from obi_notebook.get_environment import get_environment

token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects.get_projects(token)
client = Client(
    project_context=project_context,
    environment=get_environment(),
    token_manager=token,
)
# entities = get_entities.get_entities(token, project_context)
ion_channel = client.get_entity(
    entity_type=IonChannel,
    entity_id=id,
)


In [7]:
f"h.{suffix}"

'h.HCN3_0063'

In [ ]:
# create a single-compartment cell to use the ion channel model
soma = h.Section("soma")
# insert ion channel
soma.insert(f"h.{suffix}")

dict_keys(['idx', 'data_path', 'contributions', 'assets', 'license', 'creation_date', 'update_date', 'created_by', 'updated_by', 'brain_region', 'subject', 'authorized_project_id', 'authorized_public', 'type', 'id', 'experiment_date', 'contact_email', 'published_in', 'description', 'name', 'nmodl_suffix', 'is_ljp_corrected', 'is_temperature_dependent', 'temperature_celsius', 'is_stochastic', 'neuron_block'])

In [ ]:
# create an SEClamp object (to do a voltage clamp experiment) and insert it to the centre of the soma
stim = h.SEClamp(soma(0.5))

# Create vectors for recording the voltage and time.
t = h.Vector()
v = h.Vector()
t.record(h._ref_t)
v.record(soma(0.5)._ref_v)

# set the parameters of the SEClamp object.
# These values should be changed based on the channel used and experiment conditions

# Duration of different voltage clamp levels
stim.dur1 = 50
stim.dur2 = 100
stim.dur3 = 50

# Amplitude of different voltage clamp levels
stim.amp1 = -80
stim.amp2 = 0
stim.amp3 = -80

h.tstop = stim.dur1 + stim.dur2 + stim.dur3

h.run()

#plot the results

plt.plot(t, v)
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.show()